In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import csv
from matplotlib import pyplot as plt
import pandas as pd
from pathlib import Path
import torch
from torchvision import datasets, transforms
from tqdm import tqdm
import numpy as np
import random
import re
from sklearn.linear_model import LinearRegression
from scipy import stats
from scipy.stats import truncnorm

import src.architectures.networkArchitectures as netArch # Assuming this is where your registry code lives
from src.robustness.general_classes import ModelBridge
from src.robustness.milp_functions import *
from src.architectures.networkArchitectures import networkRegistry
from src.robustness.exactRobustness import exactRobustness
from src.robustness.fastLin import fastLin
from src.testing.networkTesting import testNetwork
from src.training.networkTraining import trainNetwork, saveNetwork
from src.utils.extractParams import extractParams
from src.utils.loadNetwork import loadNetwork

In [ ]:
# Train networks in registry
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

numEpochs = 2

for name, entry in networkRegistry.items():
    print(f"\nTraining {name}...")

    model = entry.NetworkClass()

    durations = trainNetwork(model, numEpochs, device)
    saveNetwork(model, name)

    print(f"Finished {name} | epoch times: {durations}")



Training Dense4x35...
Finished Dense4x35 | epoch times: [8.293872899987036, 8.303557799983537]

Training Dense1x60...
Finished Dense1x60 | epoch times: [7.65680950001115, 7.649471299984725]

Training Dense2x30...
Finished Dense2x30 | epoch times: [7.846908800012898, 7.878739899984794]

Training Dense4x30...
Finished Dense4x30 | epoch times: [8.305205300013768, 8.280840000021271]

Training Dense3x30...
Finished Dense3x30 | epoch times: [8.110198800015496, 8.145450700016227]

Training Dense6x20...
Finished Dense6x20 | epoch times: [8.695493399980478, 8.643766699999105]

Training Dense3x25...
Finished Dense3x25 | epoch times: [8.139597100001993, 8.062714300001971]

Training Dense1x65...
Finished Dense1x65 | epoch times: [7.728613999992376, 7.7233975999988616]

Training Dense3x20...
Finished Dense3x20 | epoch times: [8.058214799995767, 8.10616779999691]

Training Dense2x65...
Finished Dense2x65 | epoch times: [7.9851169000030495, 8.038287300005322]

Training Dense1x25...
Finished Dense1x2

In [4]:
DETAILED_TEST = True

for name in networkRegistry:
    print(f"----- Testing {name} -----")
    network = loadNetwork(name)
    correctClassifications, totalClassifications = testNetwork(network = network)

    if DETAILED_TEST:
        for number, correctCount in correctClassifications.items():
            accuracy = 100 * float(correctCount) / totalClassifications[number]
            print(f"Accuracy for number: {number} is {accuracy:.1f} %")

    sumOfCorrectClassifications = sum(correctClassifications.values())
    sumOfTotalClassifications = sum(totalClassifications.values())
    accuracy = 100 * float(sumOfCorrectClassifications) / sumOfTotalClassifications
    print(f"Total accuracy is {accuracy:.1f} %")

----- Testing Dense4x35 -----
Accuracy for number: 0 is 98.3 %
Accuracy for number: 1 is 98.1 %
Accuracy for number: 2 is 91.2 %
Accuracy for number: 3 is 93.2 %
Accuracy for number: 4 is 95.2 %
Accuracy for number: 5 is 91.4 %
Accuracy for number: 6 is 96.7 %
Accuracy for number: 7 is 94.7 %
Accuracy for number: 8 is 90.1 %
Accuracy for number: 9 is 89.0 %
Total accuracy is 93.8 %
----- Testing Dense1x60 -----
Accuracy for number: 0 is 98.8 %
Accuracy for number: 1 is 98.5 %
Accuracy for number: 2 is 94.8 %
Accuracy for number: 3 is 95.0 %
Accuracy for number: 4 is 95.8 %
Accuracy for number: 5 is 93.7 %
Accuracy for number: 6 is 96.2 %
Accuracy for number: 7 is 94.2 %
Accuracy for number: 8 is 93.6 %
Accuracy for number: 9 is 93.9 %
Total accuracy is 95.5 %
----- Testing Dense2x30 -----
Accuracy for number: 0 is 97.6 %
Accuracy for number: 1 is 98.2 %
Accuracy for number: 2 is 93.4 %
Accuracy for number: 3 is 93.1 %
Accuracy for number: 4 is 94.4 %
Accuracy for number: 5 is 90.7 %
Ac

In [ ]:
# Configuration
ACCURACY_THRESHOLD = 90.0
CLASSES_TO_CHECK = [0,1,3] 

performance_results = []

for name in networkRegistry:
    network = loadNetwork(name)
    correctClassifications, totalClassifications = testNetwork(network=network)
    target_classes = CLASSES_TO_CHECK if CLASSES_TO_CHECK is not None else correctClassifications.keys()    
    passed_all_classes = True
    for number in target_classes:
        accuracy = 100 * float(correctClassifications[number]) / totalClassifications[number]
        if accuracy < ACCURACY_THRESHOLD:
            passed_all_classes = False
            break
            
    performance_results.append(passed_all_classes)

# Convert to numpy array
boolean_array = np.array(performance_results)

# Print in a copy-pasteable format
print(f"np.array({repr(boolean_array.tolist())})")

np.array([True, True, True, True, True, True, False, False, True, True, True, False, False, True, False])
